```
conda create -n star_rsem_mapping
conda activate star_rsem_mapping
conda install ipykernel star rsem zstandard seqkit multiqc requests tqdm
python -m ipykernel install --user --name=star_rsem_mapping
```

# Dependencies

In [ ]:
from pathlib import Path
import subprocess
import shutil
import requests
import tqdm
import sys

# --- Import Python Utilities ---
# -------------------------------
relative_target_path = Path("utils") / "python_utils"
project_root = Path.cwd()
found_root = None
while True:
    if (project_root / relative_target_path).is_dir():
        found_root = project_root
        break # Found it!
    # Stop if we reach the filesystem root
    if project_root == project_root.parent:
        raise FileNotFoundError(
            f"Could not find the directory structure '{relative_target_path}'"
        )
    # Go one level up for the next iteration
    project_root = project_root.parent
# Add the found project root to sys.path if it's not already there
if found_root:
    path_str = str(found_root)
    if path_str not in sys.path:
        sys.path.append(path_str)
        print(f"Added '{path_str}' to sys.path")

from utils.python_utils import (
    # Directory Paths
    workflow_dir,
    log_dir,
    # Functions
    execute_command
)

# Download reference genome and annotation

In [ ]:
# --- Acomys ---
aco_cah_genome = 'https://ftp.ensembl.org/pub/rapid-release/species/Acomys_cahirinus/GCA_029890205.1/ensembl/genome/Acomys_cahirinus-GCA_029890205.1-unmasked.fa.gz'
aco_cah_annotation = 'https://ftp.ensembl.org/pub/rapid-release/species/Acomys_cahirinus/GCA_029890205.1/ensembl/geneset/2023_11/Acomys_cahirinus-GCA_029890205.1-2023_11-genes.gtf.gz'

# --- Mus ---
mus_genome = 'https://ftp.ensembl.org/pub/release-113/fasta/mus_musculus/dna/Mus_musculus.GRCm39.dna.primary_assembly.fa.gz'
mus_annotation = 'https://ftp.ensembl.org/pub/release-113/gtf/mus_musculus/Mus_musculus.GRCm39.113.gtf.gz'

# --- Output directories ---
reference_dir = Path(workflow_dir) / 'genomic_reference_data'
acomys_reference_dir = reference_dir / 'Acomys_cahirinus_reference'
mus_reference_dir = reference_dir / 'Mus_musculus_reference'

In [ ]:
def download_web_file(
        url: str,
        output_file: Path):
    try:
        with requests.get(url, stream=True) as r:
            r.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
            # Get file size from headers (in bytes)
            total_size = int(r.headers.get('content-length', 0))
            with open(output_file, 'wb') as f:
                with tqdm.tqdm(
                    total=total_size,
                    unit='B',
                    unit_scale=True,
                    unit_divisor=1024,
                    desc=f"Downloading {output_file.name}",
                    bar_format="{l_bar}{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
                ) as pbar:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                        pbar.update(len(chunk))
        if not output_file.is_file() or not output_file.stat().st_size > 0:
            print(f"Error: Failed to download {url} to {output_file}")
            return False
        print(f"Successfully downloaded {output_file}")
        return True
    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {e}")
        return False

# --- Download Mus musculus genome and cDNA ---
# ---------------------------------------------
download_success = download_web_file(
    url=mus_genome,
    output_file= Path(mus_reference_dir / 'Mus_musculus_GRCm39_primary_assembly.fasta.gz')
)
if not download_success:
    print("Failed to download Mus musculus genome")
download_success = download_web_file(
    url=mus_annotation,
    output_file=Path(mus_reference_dir / 'Mus_musculus.GRCm39.113.gtf.gz')
)
if not download_success:
    print("Failed to download Mus musculus annotation")

# --- Download Acomys genome and cDNA ---
# ---------------------------------------
download_success = download_web_file(
    url=aco_cah_genome,
    output_file=Path(acomys_reference_dir / 'Acomys_cahirinus-GCA_029890205.1-unmasked.fa.gz')
)
if not download_success:
    print("Failed to download Acomys genome")
download_success = download_web_file(
    url=aco_cah_annotation,
    output_file=Path(acomys_reference_dir / 'Acomys_cahirinus-GCA_029890205.1-2023_11-genes.gtf.gz')
)
if not download_success:
    print("Failed to download Acomys annotation")

# Create genomic index

In [ ]:
index_generate_command = [
    'STAR', 
    '--runThreadN', str(num_threads),
    '--runMode', 'genomeGenerate',
    '--genomeDir', output_dir,
    '--genomeFastaFiles', genome_file,
    '--sjdbGTFfile', annotation_file,
    '--sjdbOverhang', '59'
]

# STAR mapping
- STAR is run with `--quantMode TranscriptomeSAM` to output the transcript-level counts. The output is then quantified with [RSEM](https://github.com/deweylab/RSEM).
- 2-pass mapping protocol is used, where the first pass is used to identify novel splice junctions, and the second pass is used to map reads to the reference genome.

## 1st Pass Mapping

In [ ]:
# firstpass_command = [
#     STAR --runThreadN 30 \
#      --genomeDir "$input_index_dir" \
#      --readFilesIn R2_file.fastq.gz --readFilesCommand zcat \
#      --outFileNamePrefix "$output_prefix" --outTmpDir "$out_tmp" \
#      --outSAMtype None \
#      --outFilterMismatchNoverLmax 0.3 --outFilterMismatchNoverReadLmax 0.03 --outFilterMismatchNmax 2 \
#      --alignSJDBoverhangMin 2 --alignSJoverhangMin 8 \
#      --outFilterMultimapNmax 5 \
#      --alignIntronMin 20 --alignIntronMax 1000000 \
#      --outFilterScoreMinOverLread 0.7 --outFilterMatchNminOverLread 0.7 \
#      --winAnchorMultimapNmax 200 --seedSearchStartLmax 20 --seedSearchStartLmaxOverLread 0.4
# ]

## 2nd Pass Mapping 

In [ ]:

# STAR --runThreadN 30 \
#      --genomeDir "$input_index_dir" --sjdbFileChrStartEnd "$input_sj_file" \
#      --readFilesIn R2_file.fastq.gz --readFilesCommand zcat \
#      --outFileNamePrefix "${output_prefix}" --outTmpDir "$out_tmp" \
#      --quantMode GeneCounts TranscriptomeSAM \
#      --outFilterType BySJout \
#      --outFilterMismatchNoverLmax 0.3 --outFilterMismatchNoverReadLmax 0.03 --outFilterMismatchNmax 2 \
#      --alignSJDBoverhangMin 1 --alignSJoverhangMin 8 \
#      --outFilterMultimapNmax 3 \
#      --alignIntronMin 20 --alignIntronMax 1000000 \
#      --outFilterScoreMinOverLread 0.7 --outFilterMatchNminOverLread 0.7 \
#      --winAnchorMultimapNmax 200 --seedSearchStartLmax 20 --seedSearchStartLmaxOverLread 0.4 \

# RSEM

## Prepare reference index

In [ ]:
# rsem-prepare-reference -p 40 --gtf gtf_file \
#     genome_fasta out_path

## Run RSEM
`--estimate-rspd` for 3'-biased read start position on the trascripts (due poly-T primer single-end library preparation)

In [ ]:
# rsem-calculate-expression -p 30 --alignments \
#     --seed-length 25 \
#     --estimate-rspd \
#     --no-bam-output \
#     --strandedness forward \
#     STAR_transcriptome_output.bam \
#     "$input_rsem_ref" \
#     "$out_path" > "$log_file"